# Explore here

In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
%matplotlib inline
import sqlite3
import sqlalchemy
import json 
import os
import time
import io


url = "https://api.worldbank.org/v2"

In [3]:
pip install requests pandas matplotlib seaborn sqlalchemy


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Trabajaras con la API publica del World Bank v2 (sin autenticacion). El objetivo es analizar la evolucion socioeconomica y ambiental de 5 paises elegidos por ti entre 2010 y 2024.

Seleccion del dataset
Elige 5 paises (ISO3) que te interesen.
Elige los indicadores que quieras analizar.
Recomendacion de indicadores (opcional):
SP.POP.TOTL: Poblacion total
NY.GDP.PCAP.CD: PIB per capita (USD actuales)
EN.ATM.CO2E.PC: Emisiones de CO2 per capita (toneladas metricas)
SP.DYN.LE00.IN: Esperanza de vida al nacer (anios)
API base: https://api.worldbank.org/v2

Requisitos tecnicos
Debes trabajar en un archivo .ipynb y usar:

requests
pandas
matplotlib y/o seaborn
sqlalchemy

In [6]:
url = "https://api.worldbank.org/v2/country/ESP;THA;BRA;NGA;ITA"
response = requests.get(url)

url_base = "https://api.worldbank.org/v2"
session = requests.Session()
TIMEOUT = 60

countries = "ESP;THA;BRA;NGA;ITA"
indicators = {
    "SP.POP.TOTL":"poblacion_total",
    "NY.GDP.PCAP.CD":"PIB_per_capita",
    "EN.ATM.CO2E.PC":"emisiones_co2",
    "SP.DYN.LE00.IN":"esperanza_vida",
    "SI.POV.DDAY":"pobreza_extrema"
}

rows = []

for ind, name in indicators.items():
    url = f"https://api.worldbank.org/v2/country/{countries}/indicator/{ind}?date=2010:2024&format=json&per_page=5000"
    
    response = requests.get(url).json()
    
    if len(response) < 2:
        continue

    data = response[1]

    rows += [
        {"country": r["country"]["value"], "year": int(r["date"]), name: r["value"]}
        for r in data if r["value"] is not None
    ]

df = pd.DataFrame(rows)
df = df.groupby(["country","year"]).first().reset_index()
df = df.sort_values(["country","year"])

print(df)

     country  year  poblacion_total  PIB_per_capita  esperanza_vida  \
0     Brazil  2010      193701929.0    11403.284004          73.779   
1     Brazil  2011      195284734.0    13396.626316          74.047   
2     Brazil  2012      196876111.0    12521.723845          74.335   
3     Brazil  2013      198478299.0    12458.890340          74.609   
4     Brazil  2014      200085127.0    12274.994163          74.823   
..       ...   ...              ...             ...             ...   
70  Thailand  2020       71641484.0     6985.643939          77.331   
71  Thailand  2021       71727332.0     7057.207548          77.606   
72  Thailand  2022       71735329.0     6909.352818          75.291   
73  Thailand  2023       71702435.0     7195.101309          76.412   
74  Thailand  2024       71668011.0     7346.620221             NaN   

    pobreza_extrema  
0               NaN  
1               7.1  
2               6.4  
3               5.4  
4               4.7  
..             

In [4]:
url = "https://api.worldbank.org/v2/country/ESP;THA;BRA;NGA;ITA"
response = requests.get(url)

url_base = "https://api.worldbank.org/v2"
session = requests.Session()
TIMEOUT = 60

countries = {
    "España": "ESP",
    "Tailandia": "THA",
    "Brazil": "BRA",
    "Nigeria": "NGA",
    "Italia": "ITA"
}

indicators = {
    "población_total": "SP.POP.TOTL",
    "PIB_per_capita": "NY.GDP.PCAP.CD",
    "emisiones_co2": "EN.ATM.CO2E.PC",
    "esperanza_vida": "SP.DYN.LE00.IN",
    "pobreza_extrema": "SI.POV.DDAY"
}

def build_url(country_codes, indicator):
    countries_str = ";".join(country_codes)
    return f"{url_base}/country/{countries_str}/indicator/{indicator}?format=json&date=2010:2024&per_page=5000"

def get_latest_value(country_code, indicator_code):
    url = build_url([country_code], indicator_code)
    response = session.get(url, timeout=TIMEOUT)
    result = response.json()

    if len(result) < 2 or not result[1]:
        return None

    data = result[1]
    for row in sorted(data, key=lambda x: x["date"], reverse=True):
        if row["value"] is not None:
            return row["value"]
    return None

for country_name, country_code in countries.items():
    print(f"\n{country_name}:")
    for indicator_name, indicator_code in indicators.items():
        value = get_latest_value(country_code, indicator_code)
        print(f" - {indicator_name}: {value}")


España:
 - población_total: 48848840
 - PIB_per_capita: 35326.7683069278
 - emisiones_co2: None
 - esperanza_vida: 83.8829268292683
 - pobreza_extrema: 0.8

Tailandia:
 - población_total: 71668011
 - PIB_per_capita: 7346.62022142185
 - emisiones_co2: None
 - esperanza_vida: 76.412
 - pobreza_extrema: 0

Brazil:
 - población_total: 211998573
 - PIB_per_capita: 10310.5488778166
 - emisiones_co2: None
 - esperanza_vida: 75.848
 - pobreza_extrema: 3.8

Nigeria:
 - población_total: 232679478
 - PIB_per_capita: 1084.16041805436
 - emisiones_co2: None
 - esperanza_vida: 54.462
 - pobreza_extrema: 41.8

Italia:
 - población_total: 58952704
 - PIB_per_capita: 40385.3413957669
 - emisiones_co2: None
 - esperanza_vida: 83.7
 - pobreza_extrema: 0.9


Paso 1: Preparar entorno
Instala dependencias:

pip install requests pandas matplotlib seaborn sqlalchemy
Crea un notebook, por ejemplo: src/world_bank_analysis.ipynb.

In [5]:
results = []

for country_name, country_code in countries.items():
    for indicator_name, indicator_code in indicators.items():
        value = get_latest_value(country_code, indicator_code)
        results.append({
            "country": country_name,
            "indicator": indicator_name,
            "value": value
        })

df = pd.DataFrame(results)
df

,country,indicator,value
0,España,población_total,4.884884e+07
1,España,PIB_per_capita,3.532677e+04
2,España,emisiones_co2,NaN
3,España,esperanza_vida,8.388293e+01
4,España,pobreza_extrema,8.000000e-01
5,Tailandia,población_total,7.166801e+07
6,Tailandia,PIB_per_capita,7.346620e+03
7,Tailandia,emisiones_co2,NaN
8,Tailandia,esperanza_vida,7.641200e+01
9,Tailandia,pobreza_extrema,0.000000e+00


Paso 2: Explorar la API
Revisa estos endpoints de referencia:

Paises: https://api.worldbank.org/v2/country
Indicadores: https://api.worldbank.org/v2/indicator
Verifica la estructura de respuesta. La API pagina resultados (habitualmente hasta per_page=50), asi que debes pensar una estrategia para recorrer paginas y almacenar toda la informacion necesaria.

IMPORTANTE: El código mencionado arriba es orientativo. En el link de abajo tienes toda la información necesaria para llevar a cabo un llamado a la API:

In [6]:
df_pib = df[df["indicator"] == "PIB_per_capita"]
df_pib.sort_values("value", ascending=False)


,country,indicator,value
21,Italia,PIB_per_capita,40385.341396
1,España,PIB_per_capita,35326.768307
11,Brazil,PIB_per_capita,10310.548878
6,Tailandia,PIB_per_capita,7346.620221
16,Nigeria,PIB_per_capita,1084.160418


En este data set podemos apreciar la producto interior bruto de cada país seleccionado, podemos apreciar la diferencia entre países desarrollados como Italia o España y un país subdesarrollado como Nigeria.

In [6]:
df_vida = df[df["indicator"] == "esperanza_vida"]
df_vida.sort_values("value", ascending=False)


,country,indicator,value
3,España,esperanza_vida,83.882927
23,Italia,esperanza_vida,83.700000
8,Tailandia,esperanza_vida,76.412000
13,Brazil,esperanza_vida,75.848000
18,Nigeria,esperanza_vida,54.462000


En este data set vemos la clara diferencia entre países desarrollados como son Italia y España con una media superior a los 80 años, países en vía de desarrollo en este caso Tailandia y Brazil por encima de los 75 y un país subdesarrollado como Nigeria que no supera los 55 años de esperanza de vida.

In [7]:
df_pobreza = df[df["indicator"] == "pobreza_extrema"]
(df_pobreza.sort_values("value", ascending=False))


,country,indicator,value
19,Nigeria,pobreza_extrema,41.8
14,Brazil,pobreza_extrema,3.8
24,Italia,pobreza_extrema,0.9
4,España,pobreza_extrema,0.8
9,Tailandia,pobreza_extrema,0.0


En este caso podemos apreciar la brutal diferencia que hay entre Nigeria y los países de Europa, España e Italia.

Paso 3: Descargar datos
Descarga series temporales 2010-2024 para los paises e indicadores que elegiste.

Objetivo:

Consumir la API para varios paises e indicadores
Manejar paginacion cuando aplique
Guardar respuestas en una estructura temporal (lista de diccionarios)

In [8]:
df.to_csv("worldbank_data.csv", index=False)

df_check = pd.read_csv("worldbank_data.csv")
df_check.head()

,country,indicator,value
0,España,población_total,4.884884e+07
1,España,PIB_per_capita,3.532677e+04
2,España,emisiones_co2,NaN
3,España,esperanza_vida,8.388293e+01
4,España,pobreza_extrema,8.000000e-01


Paso 4: Transformar respuesta a DataFrames
Crea una tabla (DataFrame) por indicador para facilitar comparaciones entre paises.

Columnas sugeridas por tabla:

country
year
value
Limpieza minima:

Eliminar filas con value nulo cuando sea necesario
Convertir year a entero
Convertir value a numerico

In [9]:
def get_time_series(country_code, indicator_code):
    url = (
        f"{url_base}/country/{country_code}/indicator/{indicator_code}"
        "?format=json&date=2010:2024&per_page=5000"
    )
    response = session.get(url, timeout=TIMEOUT)
    json_data = response.json()

    if len(json_data) < 2:
        print(f"Sin datos para {country_code} - {indicator_code}")
        return []

    data = json_data[1]

    rows = []
    for row in data:
        rows.append({
            "country": row["country"]["value"],
            "year": int(row["date"]),
            "value": row["value"]
        })
    return rows

Paso 5: Analisis y visualizaciones
Genera al menos 2 graficos y explica hallazgos en celdas Markdown.

Ejemplos:

Line chart: evolucion de un indicador por pais (2010-2024)
Scatter plot: relacion entre dos indicadores para un anio reciente

In [10]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data = df[df['indicator'] == 'PIB_per_capita'],
    x = 'year',
    y = 'value',
    hue = 'country',
    marker = 'o',
    linewidth = 2
)

plt.title("Evolución del PIB per cápita (2010–2024)")
plt.ylabel("USD (precios actuales)")
plt.xlabel("Año")
plt.grid(True)
plt.legend(title="País")
plt.tight_layout()
plt.show()

ValueError: Could not interpret value `year` for `x`. An entry with this name does not appear in `data`.

<Figure size 1000x600 with 0 Axes>

In [ ]:
#ESTOY TENIENDO PROBLEMAS A LA HORA DE GRAFICAR, ME SALE COMO key error "PIB_per_capita" CUANDO LO 
#TENGO DEFINIDO EN EL PRIMER EJERCICIO Y NO SOY CAPAZ DE ENCONTRAR LA FORMA PARA QUE NO SALTE ESTE FALLO.

df_pib = tables_by_indicator["PIB_per_capita"]

plt.figure(figsize=(10,6))
sns.lineplot(
    data = df_pib,
    x = "year",
    y = "value",
    hue = "country",
    marker = 'o'
)
plt.grid(True)
plt.title("Evolución del PIB per cápita (2010–2024)")
plt.ylabel("USD")
plt.xlabel("Año")
plt.show()

KeyError: 'PIB_per_capita'

En este gráfico podemos observar la reta per capita en la selección de los píses seleccionados, vemos como los países eurpeos han obtenido una renta muy parecida con unos patrones similares, mientras los otros países mantienen un PIB por debajo al resto.

In [ ]:
#MISMO ERROR CON EL VALOR DEL DICCIONARIO "pobreza_extrema"

df_pobreza = tables_by_indicator["pobreza_extrema"]
df_last = df_pobreza.sort_values("year").groupby("country").tail(1)

plt.figure(figsize=(8,5))
sns.barplot(data=df_last, x="country", y="value")
plt.title("Pobreza extrema (% población)")
plt.ylabel("% población")
plt.xlabel("País")
plt.show()

KeyError: 'pobreza_extrema'